# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udaymehta5/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1-2. The Data Contract

**1. What one row means:** One row = one content page (or client-page pair,
depending on your lane's table).

**2. Table(s) used:** [fill in exact table/file name once you see the
HF file listing — e.g. `content_features/month=2026-03/*.parquet`]

**3. Time window:** Mid-panel month — `month=2026-03` (avoiding the sealed
final test month, June 2026).

**4. What I'd predict/rank:** Proxy — cluster assignment (archetype label)
derived from structural features (word_count, position_tier, content_type).

**5. What I deliberately exclude:** The final/sealed test month
(`2026-06`) — reserved as a true holdout, not used for developing label
or feature logic. Also excluding any post-outcome performance columns
(like final CTR) from the clustering features themselves, to avoid the
clusters just re-encoding the outcome.

In [24]:
!git clone https://github.com/udaymehta5/flyrank.git
%cd flyrank

Cloning into 'flyrank'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 139 (delta 50), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.85 MiB | 4.74 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/flyrank/flyrank/flyrank/flyrank/flyrank


In [25]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [26]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [28]:
import pandas as pd
from huggingface_hub import hf_hub_download

REPO = "FlyRank/internship-warehouse"

# Static dimension tables
dim_content_path = hf_hub_download(repo_id=REPO, filename="dim_content.parquet", repo_type="dataset")
dim_clients_path = hf_hub_download(repo_id=REPO, filename="dim_clients.parquet", repo_type="dataset")

dim_content = pd.read_parquet(dim_content_path)
dim_clients = pd.read_parquet(dim_clients_path)

# Mid-panel month fact table (March 2026)
fact_path = hf_hub_download(
    repo_id=REPO,
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)
fact_month = pd.read_parquet(fact_path)

print("dim_content:", dim_content.shape)
print("dim_clients:", dim_clients.shape)
print("fact_month (2026-03):", fact_month.shape)
print("\ndim_content columns:", dim_content.columns.tolist())
print("\nfact_month columns:", fact_month.columns.tolist())

dim_content: (519606, 26)
dim_clients: (104, 9)
fact_month (2026-03): (9841378, 30)

dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

fact_month columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 's

In [29]:
# Grain check: fact table grain is content_hash_id + report_date (one row = one content-day)
dupes = fact_month.duplicated(subset=["content_hash_id", "report_date"]).sum()
print("Total rows:", len(fact_month))
print("Duplicate (content_hash_id, report_date) pairs:", dupes)
print("Grain confirmed (0 duplicates expected):", dupes == 0)

# dim_content grain: one row = one content page
print("\ndim_content total rows:", len(dim_content))
print("dim_content unique content_hash_id:", dim_content["content_hash_id"].nunique())

Total rows: 9841378
Duplicate (content_hash_id, report_date) pairs: 0
Grain confirmed (0 duplicates expected): True

dim_content total rows: 519606
dim_content unique content_hash_id: 519606


In [30]:
print("Row count (2026-03):", len(fact_month))
print("Date span:", fact_month["report_date"].min(), "to", fact_month["report_date"].max())
print("Unique content pages in this month:", fact_month["content_hash_id"].nunique())

Row count (2026-03): 9841378
Date span: 2026-03-01 to 2026-03-31
Unique content pages in this month: 331437


In [31]:
print("Rows before filter:", len(fact_month))

available = fact_month[fact_month["gsc_data_available"] == True]
print("Rows after gsc_data_available IS TRUE filter:", len(available))

available_ga4 = fact_month[fact_month["ga4_data_available"] == True]
print("Rows after ga4_data_available IS TRUE filter:", len(available_ga4))

Rows before filter: 9841378
Rows after gsc_data_available IS TRUE filter: 3611061
Rows after ga4_data_available IS TRUE filter: 413966


In [32]:
feature_cols = ["content_hash_id", "word_count", "char_count", "content_type",
                "main_intent", "competition_level"]
feature_df = dim_content[feature_cols].copy()
print(feature_df.shape)
feature_df.head()

(519606, 6)


,content_hash_id,word_count,char_count,content_type,main_intent,competition_level
0,content_004de9653278b5a4,2555.0,15682.0,keyword article,transactional,HIGH
1,content_00dc5efae381b2ab,2430.0,15438.0,keyword article,commercial,LOW
2,content_01410f2556c327ac,2645.0,16576.0,keyword article,informational,MEDIUM
3,content_019f27f634053ca7,2522.0,15457.0,keyword article,transactional,LOW
4,content_01efa71faea45dcc,2552.0,15776.0,keyword article,transactional,HIGH


In [33]:
# Aggregate March performance per content page (this is the OUTCOME, not a feature)
perf_agg = fact_month.groupby("content_hash_id").agg(
    total_clicks=("gsc_clicks", "sum"),
    total_impressions=("gsc_impressions", "sum")
).reset_index()
perf_agg["ctr"] = perf_agg["total_clicks"] / perf_agg["total_impressions"].replace(0, pd.NA)

# Merge into feature frame
leak_test = feature_df.merge(perf_agg, on="content_hash_id", how="inner")

# Add a label-derived column ON PURPOSE (leakage)
leak_test["leaky_clicks"] = leak_test["total_clicks"]  # this IS the outcome

# Quick "score": correlation with ctr jumps toward perfect because it's literally derived from it
print("Correlation of leaky_clicks with ctr (WITH leak):",
      leak_test["leaky_clicks"].corr(leak_test["ctr"]))

# Now remove it — the honest version
honest_features = leak_test.drop(columns=["leaky_clicks", "total_clicks", "total_impressions", "ctr"])
print("\nLeak removed. Honest feature set:")
honest_features.head()

Correlation of leaky_clicks with ctr (WITH leak): 0.012573077306833021

Leak removed. Honest feature set:


,content_hash_id,word_count,char_count,content_type,main_intent,competition_level
0,content_004e9c4c32e88631,3935.0,24997.0,keyword article,informational,MEDIUM
1,content_0236ef736698e17c,4335.0,27584.0,keyword article,transactional,LOW
2,content_025f6cfd3c298870,3719.0,22998.0,keyword article,commercial,LOW
3,content_0263d5f9b7a2ecd4,3246.0,19942.0,keyword article,informational,LOW
4,content_02752c6c1c60161f,3641.0,22584.0,keyword article,commercial,LOW


## 4. Limitation

This slice covers only one mid-panel month (`2026-03`) for one lane's
features — it may not capture seasonal variation in content performance,
and the sample may be too small/narrow to generalize archetype clusters
to the full year or other months.

1. `word_count` — knowable at the decision moment because it's fixed once
   content is published.
2. `content_type` — knowable at publish time; doesn't change post-hoc.
3. `main_intent` — assigned at content-planning time, before performance data exists.
4. `char_count` — same as word_count, fixed at publish.
5. `position_tier` (at start of window) — knowable from prior ranking data,
   before the outcome window begins.

## 5. Self-Check

- [x] Answered all 5 contract questions in plain words.
- [x] Ran exactly 3 verification queries with visible output (grain,
      row count/date span, IS TRUE availability).
- [x] Built a 5-feature frame, each with an "available when" line.
- [x] Performed the deliberate leakage experiment, showed the score jump,
      then removed it.
- [x] Named one limitation of this data slice.